# CrewAI Travel Planner — Three Orchestration Styles

This class notebook compares three ways to coordinate simple AI agents:

1. **Sequential Process** — tasks run in a fixed order.
2. **Hierarchical Process** — a manager coordinates specialist agents.
3. **Flow + Router** — we explicitly define steps and conditional routes.

### Learning objective

By the end, students should be able to answer:

- Who performs the work? → **Agent**
- What work must be completed? → **Task**
- How do tasks work together? → **Process or Flow**
- How can an agent access external information? → **Tool**

> Run the notebook from top to bottom. The examples use asynchronous execution because Colab/Jupyter already runs an event loop.


## 0. Install the required libraries

`crewai-tools[serpapi]` is included because Example 2 optionally demonstrates live Google search through SerpAPI.


In [ ]:
!pip install -q crewai "crewai-tools[serpapi]"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222

## 1. Imports and API configuration

We use an OpenRouter-compatible model through CrewAI's `LLM` class.

- The **OpenRouter key** is required for all examples.
- The **SerpAPI key** is optional and used only in the hierarchical example.


In [ ]:
import os
from getpass import getpass

from crewai import Agent, Crew, LLM, Process, Task


In [ ]:
import os
from getpass import getpass

from crewai import Agent, Crew, LLM, Process, Task

# Flow imports are used in Example 3.
from crewai.flow.flow import Flow, listen, router, start

# Pydantic gives our Flow a clearly defined state structure.
from pydantic import BaseModel


# Never write secret keys directly in a shared notebook.
openrouter_key = getpass("Enter your OpenRouter API key: ")

os.environ["OPENAI_API_KEY"] = openrouter_key
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"


# One LLM object can be reused by several agents.
llm = LLM(
    model="openai/gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    temperature=0.3,
    max_tokens=150,
)

print("LLM configured successfully.")


Enter your OpenRouter API key: ··········
LLM configured successfully.


# Example 1 — Sequential Crew

## Idea

A sequential crew is appropriate when the order is already known.

```text
Research → Build itinerary → Estimate budget
```

The tasks are executed according to their order in the `tasks` list.

### Classroom analogy

Think of an assembly line:

1. The researcher prepares information.
2. The planner uses that information.
3. The budget advisor checks the completed plan.


In [ ]:
def build_sequential_crew(destination: str, days: int, budget_style: str) -> Crew:
    """Create a fixed, step-by-step travel-planning crew."""

    # AGENT 1: responsible for destination research.
    researcher = Agent(
        role="Destination Researcher",
        goal=f"Collect useful travel information about {destination}",
        backstory="You summarize attractions, food, culture, and transport clearly.",
        llm=llm,
        verbose=True,
    )

    # AGENT 2: turns research into a daily schedule.
    planner = Agent(
        role="Itinerary Planner",
        goal=f"Create a practical {days}-day itinerary for {destination}",
        backstory="You organize activities by day and avoid unrealistic travel times.",
        llm=llm,
        verbose=True,
    )

    # AGENT 3: estimates costs after seeing the itinerary.
    budget_advisor = Agent(
        role="Budget Advisor",
        goal=f"Estimate costs for a {budget_style} trip",
        backstory="You create simple travel budgets and suggest savings.",
        llm=llm,
        verbose=True,
    )

    # TASK 1 runs first.
    research_task = Task(
        description=(
            f"Research {destination} for a {days}-day visit. Include key attractions, "
            "local food, transport, and two important travel tips."
        ),
        expected_output="A concise destination research summary.",
        agent=researcher,
    )

    # TASK 2 uses Task 1's output as context.
    itinerary_task = Task(
        description=(
            f"Create a {days}-day itinerary for {destination}. For each day, provide "
            "Morning, Afternoon, and Evening activities."
        ),
        expected_output="A clear day-wise itinerary.",
        agent=planner,
        context=[research_task],
    )

    # TASK 3 uses the earlier research and itinerary outputs.
    budget_task = Task(
        description=(
            f"Estimate accommodation, food, local transport, and activity costs for "
            f"the itinerary using a {budget_style} travel style. State the currency."
        ),
        expected_output="A category-wise cost estimate and total budget.",
        agent=budget_advisor,
        context=[research_task, itinerary_task],
    )

    # Process.sequential means CrewAI follows the task-list order.
    return Crew(
        agents=[researcher, planner, budget_advisor],
        tasks=[research_task, itinerary_task, budget_task],
        process=Process.sequential,
        verbose=True,
    )


In [ ]:
# Run Example 1
sequential_crew = build_sequential_crew(
    destination="Bali",
    days=3,
    budget_style="moderate",
)

sequential_result = await sequential_crew.kickoff_async()

print("\n========== SEQUENTIAL RESULT ==========\n")
print(sequential_result.raw)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 124fe748-e1da-4fbf-a80d-88b744908da0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research Bali for a 3-day visit. Include key attractions, local food, transport, and two important       │
│  travel tips.                                                                                                   │
│  ID: 899cdc69-a29a-430f-be73-f1e229b15ee2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Destination Researcher                                                                                  │
│                                                                                                                 │
│  Task: Research Bali for a 3-day visit. Include key attractions, local food, transport, and two important       │
│  travel tips.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Destination Researcher                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Destination Research Summary: Bali for a 3-Day Visit**                                                       │
│                                                                                                                 │
│  **Key Attractions:**                                                                                           │
│                                                                                                                 │
│  1. **Uluwatu Temple**: Perched on a cliff, this iconic sea temple offers stunning ocean views and is famous    │
│  for its Kecak dance performances at sunset.                                                                    │
│                                                                                                                 │
│  2. **Tegallalang Rice Terraces**: Located near Ubud, these picturesque rice paddies are a must-see for their   │
│  breathtaking landscapes and are perfect for photography.                                                       │
│                                                                                                                 │
│  3. **Sacred Monkey Forest Sanctuary**: Also in Ubud, this sanctuary is home to hundreds of playful monkeys     │
│  and ancient temples, providing a unique experience with nature and culture.                                    │
│                                                                                                                 │
│  4. **Tanah Lot Temple**: Another stunning sea temple, Tanah Lot is best visited during sunset                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Budget Advisor                                                                                          │
│                                                                                                                 │
│  Task: Estimate accommodation, food, local transport, and activity costs for the itinerary using a moderate     │
│  travel style. State the currency.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Budget Advisor                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Travel Budget Estimate for a 3-Day Visit to Bali (Currency: Indonesian Rupiah - IDR)**                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Accommodation:**                                                                                             │
│  - Average cost for a mid-range hotel: IDR 800,000 per night                                                    │
│  - Total for 3 nights: IDR 2,400,000                                                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Food:**                                                                                                      │
│  - Breakfast: IDR 50,000 per meal (3 days) = IDR 150,000                                                        │
│  - Lunch: IDR 100,000 per meal (3 days) = IDR 300,000                                                           │
│  - Dinner: IDR 150,000 per meal (3 days) = IDR 450,000                                                          │
│  - Total for food: IDR 900,000                                                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **                                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Estimate accommodation, food, local transport, and activity costs for the itinerary using a moderate     │
│  travel style. State the currency.                                                                              │
│  Agent: Budget Advisor                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


========== SEQUENTIAL RESULT ==========

**Travel Budget Estimate for a 3-Day Visit to Bali (Currency: Indonesian Rupiah - IDR)**

---

**Accommodation:**
- Average cost for a mid-range hotel: IDR 800,000 per night
- Total for 3 nights: IDR 2,400,000

---

**Food:**
- Breakfast: IDR 50,000 per meal (3 days) = IDR 150,000
- Lunch: IDR 100,000 per meal (3 days) = IDR 300,000
- Dinner: IDR 150,000 per meal (3 days) = IDR 450,000
- Total for food: IDR 900,000

---

**


## What students should notice

- The execution order is predictable.
- Each task has a specific agent.
- `context=[...]` passes previous task outputs to later tasks.
- There is no manager deciding who should work next.


# Example 2 — Hierarchical Crew with an Optional SerpAPI Tool

## Idea

A hierarchical crew behaves more like a team managed by a supervisor.

```text
                 Travel Manager
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
     Researcher     Planner     Budget Advisor
```

The manager coordinates and delegates work. We still provide tasks, but the manager controls collaboration.

## Tool concept

An LLM knows patterns from its training data, but it does not automatically perform a live Google search. A **tool** gives an agent an external capability.

In this example, the researcher can optionally receive `SerpApiGoogleSearchTool`.


In [ ]:
# -----------------------------------------
# OPTIONAL SERPAPI TOOL
# -----------------------------------------

import os
from getpass import getpass
from crewai.tools import tool


# Leave the key blank to run without live web search.
serpapi_key = getpass(
    "Enter SerpAPI key, or press Enter to skip: "
).strip()


# By default, the researcher has no web-search tools.
research_tools = []


if serpapi_key:
    os.environ["SERPAPI_API_KEY"] = serpapi_key

    from crewai_tools import SerpApiGoogleSearchTool

    search_tool = SerpApiGoogleSearchTool()

    # This tool will be given to the researcher.
    research_tools = [search_tool]

    print("SerpAPI search tool enabled.")

else:
    print(
        "SerpAPI skipped. "
        "The researcher will use only the LLM."
    )


# -----------------------------------------
# CUSTOM TOOL EXAMPLE
# -----------------------------------------

@tool("Travel Budget Calculator")
def calculate_travel_budget(
    accommodation_per_day: float,
    food_per_day: float,
    transport_per_day: float,
    activity_cost: float,
    days: int
) -> str:
    """
    Calculate the estimated total travel cost.

    Use this tool when accommodation, food, transport,
    activity cost, and number of days are available.

    Args:
        accommodation_per_day:
            Estimated accommodation cost for one day.

        food_per_day:
            Estimated food cost for one day.

        transport_per_day:
            Estimated local transport cost for one day.

        activity_cost:
            Total activity cost for the complete trip.

        days:
            Total number of travel days.

    Returns:
        A formatted category-wise travel budget.
    """

    # Calculate the cost for the full trip.
    accommodation_total = accommodation_per_day * days
    food_total = food_per_day * days
    transport_total = transport_per_day * days

    total_cost = (
        accommodation_total
        + food_total
        + transport_total
        + activity_cost
    )

    # Return readable text for the agent.
    return f"""
Travel Budget Calculation

Accommodation:
{accommodation_per_day:.2f} × {days} days
= {accommodation_total:.2f}

Food:
{food_per_day:.2f} × {days} days
= {food_total:.2f}

Local transport:
{transport_per_day:.2f} × {days} days
= {transport_total:.2f}

Activities:
{activity_cost:.2f}

Total estimated cost:
{total_cost:.2f}
""".strip()


# This list will be given to the budget advisor.
budget_tools = [calculate_travel_budget]

print("Custom travel budget tool enabled.")

Enter SerpAPI key, or press Enter to skip: ··········
SerpAPI search tool enabled.
Custom travel budget tool enabled.


In [ ]:
def build_hierarchical_crew(
    destination: str,
    days: int,
    budget_style: str
) -> Crew:
    """
    Create a manager-led travel crew.

    Tool access:
    - Researcher: optional SerpAPI search tool
    - Planner: no tool
    - Budget advisor: custom budget calculator
    """

    # -----------------------------------------
    # SPECIALIST 1: RESEARCHER
    # -----------------------------------------

    researcher = Agent(
        role="Destination Researcher",

        goal=(
            f"Find useful and accurate travel information "
            f"about {destination}"
        ),

        backstory=(
            "You are a travel researcher who investigates "
            "attractions, food, culture, transport, and practical "
            "travel information. Use web search when it is available."
        ),

        # [] when SerpAPI was skipped.
        # [search_tool] when SerpAPI was enabled.
        tools=research_tools,

        llm=llm,
        verbose=True,
    )

    # -----------------------------------------
    # SPECIALIST 2: ITINERARY PLANNER
    # -----------------------------------------

    planner = Agent(
        role="Itinerary Planner",

        goal=(
            f"Design a practical {days}-day plan "
            f"for {destination}"
        ),

        backstory=(
            "You convert destination research into a balanced "
            "daily itinerary. You group nearby attractions together "
            "and avoid unrealistic travel schedules."
        ),

        llm=llm,
        verbose=True,
    )

    # -----------------------------------------
    # SPECIALIST 3: BUDGET ADVISOR
    # -----------------------------------------

    budget_advisor = Agent(
        role="Budget Advisor",

        goal=(
            f"Keep the travel plan appropriate for "
            f"a {budget_style} traveler"
        ),

        backstory=(
            "You estimate accommodation, food, transport, "
            "and activity expenses. You use the budget calculator "
            "to verify the mathematical total."
        ),

        # The custom tool is available only to this agent.
        tools=budget_tools,

        llm=llm,
        verbose=True,
    )

    # -----------------------------------------
    # MANAGER AGENT
    # -----------------------------------------

    manager = Agent(
        role="Travel Manager",

        goal=(
            "Coordinate the specialists and produce one "
            "consistent travel plan"
        ),

        backstory=(
            "You delegate work, review specialist outputs, "
            "identify missing information, and resolve gaps."
        ),

        llm=llm,

        # Allows the manager to delegate work.
        allow_delegation=True,

        verbose=True,
    )

    # -----------------------------------------
    # TASK 1: DESTINATION RESEARCH
    # -----------------------------------------

    research_task = Task(
        description=(
            f"Research attractions, food, transport, culture, "
            f"and important travel tips for {destination}. "
            "Use the web-search tool when it is available and useful."
        ),

        expected_output=(
            "A concise destination briefing containing attractions, "
            "food recommendations, transport advice, and travel tips."
        ),

        agent=researcher,
    )

    # -----------------------------------------
    # TASK 2: ITINERARY
    # -----------------------------------------

    itinerary_task = Task(
        description=(
            f"Using the destination research, prepare a realistic "
            f"{days}-day itinerary for {destination}. "
            "Divide each day into Morning, Afternoon, and Evening. "
            "Group nearby activities together."
        ),

        expected_output=(
            "A practical day-wise itinerary with Morning, "
            "Afternoon, and Evening activities."
        ),

        agent=planner,

        # The planner receives the research result.
        context=[research_task],
    )

    # -----------------------------------------
    # TASK 3: BUDGET
    # -----------------------------------------

    budget_task = Task(
        description=(
            f"Review the research and itinerary and estimate the cost "
            f"of the trip for a {budget_style} traveler.\n\n"
            "Provide reasonable estimated values for:\n"
            "- Accommodation per day\n"
            "- Food per day\n"
            "- Local transport per day\n"
            "- Total activity cost\n\n"
            "Use the Travel Budget Calculator tool to calculate and "
            "verify the final total. State the currency and provide "
            "three money-saving suggestions."
        ),

        expected_output=(
            "A category-wise cost breakdown, verified total trip cost, "
            "currency, and money-saving recommendations."
        ),

        agent=budget_advisor,

        # The budget advisor receives both earlier outputs.
        context=[research_task, itinerary_task],
    )

    # -----------------------------------------
    # CREATE THE CREW
    # -----------------------------------------

    return Crew(
        # Specialist agents available in the crew.
        agents=[
            researcher,
            planner,
            budget_advisor,
        ],

        # Tasks to be coordinated by the manager.
        tasks=[
            research_task,
            itinerary_task,
            budget_task,
        ],

        # The manager supervises the specialists.
        manager_agent=manager,

        # Activates manager-led coordination.
        process=Process.hierarchical,

        verbose=True,
    )

In [ ]:
destination = "Bali"
days = 4
budget_style = "moderate"

crew = build_hierarchical_crew(
    destination=destination,
    days=days,
    budget_style=budget_style,
)

result = await crew.kickoff_async()

print("\n\n================ FINAL TRIP PLAN ================\n")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fa6539ab-4276-408d-82ad-d8d252f977d6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research attractions, food, transport, culture, and important travel tips for Bali. Use the web-search   │
│  tool when it is available and useful.                                                                          │
│  ID: 5c3861e7-d491-4333-81b0-35c0a36972ed                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Manager                                                                                          │
│                                                                                                                 │
│  Task: Research attractions, food, transport, culture, and important travel tips for Bali. Use the web-search   │
│  tool when it is available and useful.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Research attractions, food, transport, culture, and important travel tips for Bali. Use the    │
│  web-search tool when it is available and useful.', 'context': 'You need to gather comprehensive in...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Destination Researcher                                                                                  │
│                                                                                                                 │
│  Task: Research attractions, food, transport, culture, and important travel tips for Bali. Use the web-search   │
│  tool when it is available and useful.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Manager                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Travel Budget for Bali Trip                                                                                │
│                                                                                                                 │
│  #### Cost Breakdown                                                                                            │
│                                                                                                                 │
│  - **Accommodation**:                                                                                           │
│    - Estimated Cost: $50.00 per day                                                                             │
│    - Total for 4 days: $200.00                                                                                  │
│                                                                                                                 │
│  - **Food**:                                                                                                    │
│    - Estimated Cost: $20.00 per day                                                                             │
│    - Total for 4 days: $80.00                                                                                   │
│                                                                                                                 │
│  - **Local Transport**:                                                                                         │
│    - Estimated Cost: $15.00 per day                                                                             │
│    - Total for 4 days: $60.00                                                                                   │
│                                                                                                                 │
│  - **Activities**:                                                                                              │
│    - Total Activity Cost: $30.00                                                                                │
│                                                                                                                 │
│  #### Total Estimated Cost                                                                                      │
│  - **Total Trip Cost**: $370.00                                                                                 │
│                                                                                                                 │
│  #### Currency                                                                                                  │
│  - **Currency**: USD (                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the research and itinerary and estimate the cost of the trip for a moderate traveler.             │
│                                                                                                                 │
│  Provide reasonable estimated values for:                                                                       │
│  - Accommodation per day                                                                                        │
│  - Food per day                                                                                                 │
│  - Local transport per day                                                                                      │
│  - Total activity cost                                                                                          │
│                                                                                                                 │
│  Use the Travel Budget Calculator tool to calculate and verify the final total. State the currency and provide  │
│  three money-saving suggestions.                                                                                │
│  Agent: Travel Manager                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



================ FINAL TRIP PLAN ================

### Travel Budget for Bali Trip

#### Cost Breakdown

- **Accommodation**: 
  - Estimated Cost: $50.00 per day
  - Total for 4 days: $200.00

- **Food**: 
  - Estimated Cost: $20.00 per day
  - Total for 4 days: $80.00

- **Local Transport**: 
  - Estimated Cost: $15.00 per day
  - Total for 4 days: $60.00

- **Activities**: 
  - Total Activity Cost: $30.00

#### Total Estimated Cost
- **Total Trip Cost**: $370.00

#### Currency
- **Currency**: USD (


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: fa6539ab-4276-408d-82ad-d8d252f977d6                                                                       │
│  Final Output: ### Travel Budget for Bali Trip                                                                  │
│                                                                                                                 │
│  #### Cost Breakdown                                                                                            │
│                                                                                                                 │
│  - **Accommodation**:                                                                                           │
│    - Estimated Cost: $50.00 per day                                                                             │
│    - Total for 4 days: $200.00                                                                                  │
│                                                                                                                 │
│  - **Food**:                                                                                                    │
│    - Estimated Cost: $20.00 per day                                                                             │
│    - Total for 4 days: $80.00                                                                                   │
│                                                                                                                 │
│  - **Local Transport**:                                                                                         │
│    - Estimated Cost: $15.00 per day                                                                             │
│    - Total for 4 days: $60.00                                                                                   │
│                                                                                                                 │
│  - **Activities**:                                                                                              │
│    - Total Activity Cost: $30.00                                                                                │
│                                                                                                                 │
│  #### Total Estimated Cost                                                                                      │
│  - **Total Trip Cost**: $370.00                                                                                 │
│                                                                                                                 │
│  #### Currency                                                                                                  │
│  - **Currency**: USD (                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## What students should notice

- The manager is separate from the specialist-agent list.
- `Process.hierarchical` gives the manager coordination responsibility.
- A tool is attached to an **agent** through `tools=[...]`.
- Giving an agent a tool does not guarantee it will call the tool every time; the agent decides when it is useful.


# Example 3 — CrewAI Flow with Explicit Steps and a Router

## Idea

A Flow is useful when the developer wants to define the path explicitly.

```text
Start: create draft itinerary
              │
              ▼
         Route by budget
          ┌───────────┐
          ▼           ▼
   Budget route   Premium route
```

Here:

- `@start()` defines the first step.
- `@router()` returns a route name.
- `@listen("route_name")` runs only for that route.
- State stores information shared by the Flow.

Unlike the hierarchical example, the route is controlled by Python logic rather than by a manager agent.


In [ ]:
class TravelFlowState(BaseModel):
    """Data shared between Flow steps."""

    destination: str = "Bali"
    days: int = 3
    budget_style: str = "moderate"
    draft_plan: str = ""
    final_plan: str = ""


class TravelPlanningFlow(Flow[TravelFlowState]):
    """A small Flow demonstrating explicit steps and conditional routing."""

    @start()
    async def create_draft(self):
        """STEP 1: A simple agent creates the initial itinerary."""

        planner = Agent(
            role="Travel Planner",
            goal=f"Create a simple trip plan for {self.state.destination}",
            backstory="You create short and practical itineraries.",
            llm=llm,
            verbose=True,
        )

        task = Task(
            description=(
                f"Create a {self.state.days}-day draft itinerary for "
                f"{self.state.destination}. Keep it concise."
            ),
            expected_output="A short day-wise draft itinerary.",
            agent=planner,
        )

        # A one-agent crew is used inside this Flow step.
        crew = Crew(
            agents=[planner],
            tasks=[task],
            process=Process.sequential,
            verbose=True,
        )

        result = await crew.kickoff_async()
        self.state.draft_plan = result.raw

        print("STEP 1 COMPLETE: Draft itinerary created.")
        return self.state.draft_plan

    @router(create_draft)
    def choose_budget_route(self):
        """STEP 2: Python chooses the next path."""

        style = self.state.budget_style.lower().strip()

        if style in {"luxury", "premium"}:
            print("ROUTER DECISION: premium_route")
            return "premium_route"

        print("ROUTER DECISION: budget_route")
        return "budget_route"

    @listen("budget_route")
    async def optimize_for_budget(self):
        """ROUTE A: Runs for budget or moderate travel styles."""

        advisor = Agent(
            role="Savings Advisor",
            goal="Reduce travel cost without removing the main experiences",
            backstory="You find practical ways to save money while traveling.",
            llm=llm,
            verbose=True,
        )

        task = Task(
            description=(
                f"Improve this itinerary for a {self.state.budget_style} traveler. "
                "Add approximate category costs and three saving tips.\n\n"
                f"DRAFT ITINERARY:\n{self.state.draft_plan}"
            ),
            expected_output="A budget-conscious final itinerary with cost guidance.",
            agent=advisor,
        )

        crew = Crew(agents=[advisor], tasks=[task], process=Process.sequential)
        result = await crew.kickoff_async()
        self.state.final_plan = result.raw
        return self.state.final_plan

    @listen("premium_route")
    async def enhance_for_premium(self):
        """ROUTE B: Runs only for luxury or premium travel styles."""

        concierge = Agent(
            role="Luxury Travel Concierge",
            goal="Upgrade a trip with comfortable and premium experiences",
            backstory="You recommend high-quality stays, dining, and private transport.",
            llm=llm,
            verbose=True,
        )

        task = Task(
            description=(
                "Upgrade the following draft with premium accommodation, dining, "
                "transport, and one special experience.\n\n"
                f"DRAFT ITINERARY:\n{self.state.draft_plan}"
            ),
            expected_output="A premium final itinerary with upgraded experiences.",
            agent=concierge,
        )

        crew = Crew(agents=[concierge], tasks=[task], process=Process.sequential)
        result = await crew.kickoff_async()
        self.state.final_plan = result.raw
        return self.state.final_plan


In [ ]:
# Run Example 3
# Change budget_style to "luxury" to test the other route.

travel_flow = TravelPlanningFlow()

flow_result = await travel_flow.kickoff_async(
    inputs={
        "destination": "Bali",
        "days": 3,
        "budget_style": "moderate",
    }
)

print("\n========== FLOW RESULT ==========\n")
print(flow_result)


╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: TravelPlanningFlow                                                                                       │
│  ID: 7691c48b-e446-4d7b-870f-e5f0881215f0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: TravelPlanningFlow                                                                                       │
│  ID: 7691c48b-e446-4d7b-870f-e5f0881215f0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: create_draft                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 95011820-dda3-4f00-8c64-78900b33623c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a 3-day draft itinerary for Bali. Keep it concise.                                                │
│  ID: b9d91ce9-6ec8-4215-bb8b-64f5b8378d06                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Savings Advisor                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **3-Day Bali Itinerary**                                                                                       │
│                                                                                                                 │
│  **Day 1: Ubud Exploration**                                                                                    │
│  - **Morning:**                                                                                                 │
│    - Breakfast at a local café (try clear soup or smoothie bowls).                                              │
│      *Approximate Cost: IDR 50,000 - 100,000 ($3.50 - $7)*                                                      │
│    - Visit the Sacred Monkey Forest Sanctuary.                                                                  │
│      *Approximate Cost: IDR 80,000 ($5.50)*                                                                     │
│                                                                                                                 │
│  - **Afternoon:**                                                                                               │
│    - Lunch at a restaurant overlooking rice terraces (e.g., Warung Babi Guling).                                │
│      *Approximate Cost: IDR 100,000 ($7)*                                                                       │
│    - Explore Tegalalang Rice Terraces and take photos.                                                          │
│      *                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: optimize_for_budget                                                                                    │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: TravelPlanningFlow                                                                                       │
│  ID: 7691c48b-e446-4d7b-870f-e5f0881215f0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


========== FLOW RESULT ==========

**3-Day Bali Itinerary**

**Day 1: Ubud Exploration**
- **Morning:**
  - Breakfast at a local café (try clear soup or smoothie bowls).  
    *Approximate Cost: IDR 50,000 - 100,000 ($3.50 - $7)*  
  - Visit the Sacred Monkey Forest Sanctuary.  
    *Approximate Cost: IDR 80,000 ($5.50)*

- **Afternoon:**
  - Lunch at a restaurant overlooking rice terraces (e.g., Warung Babi Guling).  
    *Approximate Cost: IDR 100,000 ($7)*  
  - Explore Tegalalang Rice Terraces and take photos.  
    *


## Router experiment for students

Run the Flow twice:

```python
"budget_style": "moderate"
```

Then change it to:

```python
"budget_style": "luxury"
```

Observe that the first agent still creates the draft, but the router chooses a different final agent.


# Final Comparison

| Option | Who controls the steps? | Best use | Travel example |
|---|---|---|---|
| Sequential Crew | Task-list order | Simple fixed pipelines | Research → itinerary → budget |
| Hierarchical Crew | Manager agent | Delegation and collaboration | Travel manager coordinates specialists |
| Flow + Router | Developer's Python logic | Exact paths, conditions, loops | Moderate route vs premium route |

## One-line mental mapping aid

- **Sequential:** “Follow these tasks in order.”
- **Hierarchical:** “Let the manager coordinate the team.”
- **Flow:** “I will explicitly define what happens next.”


In [ ]:
# Assignment 1: AI Career Advisor
# Problem Statement
# Develop an Agentic AI Career Advisor using CrewAI that helps students choose a suitable career path based on their interests and skills.
# Required Agents
# Career Analyst
# -Analyze user skills
# -Identify strengths
# Job Market Researcher
# -Research trending jobs
# -Mention required skills
# Learning Roadmap Planner
# -Suggest courses
# -Suggest certifications
# -Create a learning roadmap
# Resume Reviewer
# -Suggest resume improvements
# -Highlight missing skills


# Inputs
# Skills:
# Python
# SQL
# Machine Learning

# Interest:
# Generative AI

# Experience:
# 1 year


# Output
# Suitable career
# Skills gap
# Learning roadmap
# Resume suggestions
# Final recommendation
# Requirements

# Use
# Sequential Process
#  OR
# Hierarchical Process
# Students should justify why they selected that process.
